# Human Sign-off for Irreversible Tool Calls

Grok's function calling lets an agent *act* — look things up, call APIs, move data. For most calls that's exactly what you want. But some actions are **irreversible**: sending a payment, changing a payout account, deleting records, deploying code. For those you often want a **named human to approve before the action runs** — without slowing down everything else.

This cookbook adds a minimal human-in-the-loop step on top of [function calling](../function_calling_101/guide.ipynb):

1. **Classify** each tool call — most are low-risk and run immediately.
2. **Route** irreversible ones through an approval step (`allow` / `signoff_required` / `deny`).
3. **Execute** only once a human has approved.

We use a tiny local policy so the whole notebook runs with just your xAI API key. The last section shows how to take it to production.

## Setup

Grok is OpenAI-compatible, so we use the same `openai` client and `chat.completions` API as `function_calling_101`.

In [1]:
%pip install openai


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /private/tmp/nbvenv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json
from openai import OpenAI

# Grok via the OpenAI-compatible endpoint (set XAI_API_KEY in your environment).
client = OpenAI(api_key=os.environ["XAI_API_KEY"], base_url="https://api.x.ai/v1")
MODEL = "grok-4.3"

## 1. Tools — one read-only, one irreversible

`get_invoice` just reads. `release_payment` moves money — that's the one we want a human to sign off on for large amounts.

In [3]:
def get_invoice(invoice_id: str):
    """Read-only: look up an invoice."""
    return {"invoice_id": invoice_id, "amount_due": 82000, "vendor": "acct_9f12"}

def release_payment(amount: float, destination: str):
    """IRREVERSIBLE: release a wire/ACH payment."""
    # A real implementation would move money here.
    return {"status": "released", "amount": amount, "destination": destination}

tools_map = {"get_invoice": get_invoice, "release_payment": release_payment}

tools_definition = [
    {"type": "function", "function": {
        "name": "get_invoice",
        "description": "Look up an invoice (read-only).",
        "parameters": {"type": "object",
            "properties": {"invoice_id": {"type": "string"}},
            "required": ["invoice_id"]}}},
    {"type": "function", "function": {
        "name": "release_payment",
        "description": "Release a wire/ACH payment. IRREVERSIBLE — money leaves the account.",
        "parameters": {"type": "object",
            "properties": {"amount": {"type": "number"}, "destination": {"type": "string"}},
            "required": ["amount", "destination"]}}},
]

## 2. The guard — which calls need a human

A small, transparent policy. Read-only calls run freely; an irreversible payment over a threshold (or to a blocked destination) returns a decision the loop will act on. Keeping the decision shape — `allow` / `signoff_required` / `deny` — lets you swap in a hosted policy engine later without changing the loop.

In [4]:
IRREVERSIBLE = {"release_payment"}
SIGNOFF_THRESHOLD = 50_000

def guard(tool_name: str, args: dict) -> dict:
    if tool_name not in IRREVERSIBLE:
        return {"decision": "allow"}                       # read-only / low-risk
    if "sanctioned" in str(args.get("destination", "")):
        return {"decision": "deny", "reason": "destination on blocklist"}
    if float(args.get("amount", 0)) >= SIGNOFF_THRESHOLD:
        return {"decision": "signoff_required",
                "reason": f"payment >= ${SIGNOFF_THRESHOLD:,}"}
    return {"decision": "allow"}

In [5]:
# The policy in action (no model call needed):
for name, args in [
    ("get_invoice",     {"invoice_id": "INV-1"}),
    ("release_payment", {"amount": 30,    "destination": "acct_a"}),
    ("release_payment", {"amount": 82000, "destination": "acct_b"}),
    ("release_payment", {"amount": 5000,  "destination": "acct_sanctioned"}),
]:
    print(f"{name}({args}) -> {guard(name, args)['decision']}")

get_invoice({'invoice_id': 'INV-1'}) -> allow
release_payment({'amount': 30, 'destination': 'acct_a'}) -> allow
release_payment({'amount': 82000, 'destination': 'acct_b'}) -> signoff_required
release_payment({'amount': 5000, 'destination': 'acct_sanctioned'}) -> deny


## 3. The approval step

This is the integration point for a *named* human's decision. In production they approve out-of-band (a dashboard or Slack) and the action resumes asynchronously, so high-volume flows never block on a person. For this runnable notebook we auto-approve.

In [6]:
def request_human_signoff(tool_name: str, args: dict, reason: str) -> bool:
    """In production a NAMED human approves out-of-band (dashboard / Slack) and the
    action resumes asynchronously. For this runnable demo we auto-approve and print
    what the human would see."""
    print(f"  sign-off required: {tool_name}({args}) -- {reason}")
    print("  (demo: auto-approving; wire this to your real approval UI in production)")
    return True

## 4. The guarded tool-calling loop

This is `function_calling_101`'s handler with one addition: before running a tool call, we check `guard()` and route irreversible ones through approval.

In [7]:
def run_guarded(prompt: str, max_turns: int = 6):
    chat_history = [
        {"role": "system", "content": "You are an autonomous treasury assistant. Use the tools to carry out the request; do not ask the user to confirm."},
        {"role": "user", "content": prompt},
    ]
    for _ in range(max_turns):
        msg = client.chat.completions.create(
            model=MODEL, messages=chat_history,
            tools=tools_definition, tool_choice="auto").choices[0].message
        chat_history.append(msg.to_dict())

        if not msg.tool_calls:
            print("\n" + (msg.content or ""))
            return

        for tool_call in msg.tool_calls:
            name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            decision = guard(name, args)

            if decision["decision"] == "deny":
                result = {"error": f"blocked: {decision['reason']}"}
            elif decision["decision"] == "signoff_required":
                approved = request_human_signoff(name, args, decision["reason"])
                result = tools_map[name](**args) if approved else {"error": "human sign-off declined"}
            else:
                result = tools_map[name](**args)

            print(f"  {name}({args}) -> {decision['decision']}")
            chat_history.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})

## 5. Try it

A small refund runs straight through. A large payment stops for a human.

In [8]:
# Small — runs immediately
run_guarded("Refund $30 to the customer on order 5510.")

  get_invoice({'invoice_id': '5510'}) -> allow


  release_payment({'amount': 30.0, 'destination': 'acct_9f12'}) -> allow



Refund of $30 processed successfully to acct_9f12 for order 5510.


In [9]:
# Large — requires sign-off (answer y/N at the prompt)
run_guarded("Pay invoice INV-4421 in full to the vendor.")

  get_invoice({'invoice_id': 'INV-4421'}) -> allow


  sign-off required: release_payment({'amount': 82000.0, 'destination': 'acct_9f12'}) -- payment >= $50,000
  (demo: auto-approving; wire this to your real approval UI in production)
  release_payment({'amount': 82000.0, 'destination': 'acct_9f12'}) -> signoff_required



Payment released for INV-4421: $82,000.00 to acct_9f12.


## Taking this to production

The policy above is a few lines for clarity. A real deployment usually wants:

- a **policy engine** you can audit and version, so *which rule applied* is provable after the fact;
- **offline-verifiable receipts** of each decision — who approved what, checkable without calling home;
- **async, out-of-band approval** so high-volume flows aren't blocked on a human.

[EMILIA Protocol](https://github.com/emiliaprotocol/emilia-protocol) is one open-source (Apache-2.0) implementation of this pattern — a drop-in guard for OpenAI-compatible clients (Grok included) plus a reproducible cross-model benchmark. The local `guard()` above mirrors its `allow` / `signoff_required` / `deny` decision shape, so swapping in a hosted engine is a one-line change.